# Simple Parallel Workflow : 

Without LLm 

In [36]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict

In [37]:
# Define State : 

class BatsmanState(TypedDict):

     runs  : int # runs are same for all parallel tasks/nodes so dont pass it as parameter.
     balls : int # balls are same for all parallel tasks/nodes so dont pass it as parameter
     fours : int
     sixes : int 
     sr  : float # strike rate 
     bpb : float # balls  per boundary 
     boundary_percent : float # biundary percentage
     summary : str

In [38]:
# Tasks / Functions : 


# IMP :
# runs are same for all parallel tasks/nodes so dont return entire state again just return the variable in which changes are made
# balls are same for all parallel tasks/nodes so so dont return entire state again just return the variable in which changes are made


# Task 1 : 

def calculate_sr(state : BatsmanState):
    sr =  (state['runs']/state['balls'])*100

    state['sr'] = sr

    return {'sr' : sr}

In [39]:
# Task 2 : 

def calculate_bpb(state : BatsmanState):
    bpb =  (state['balls']/(state['fours']+state['sixes']))/100

    state['bpb'] = bpb

    return {'bpb' : bpb}

In [40]:
# Task 3 : 

def calculate_boundary_percent(state : BatsmanState):
    boundary_percent =  (((state['fours']*4 + state['sixes']*6)/state['runs']))*100

    state['boundary_percent'] = boundary_percent

    return {'boundary_percent' : boundary_percent}

In [41]:
# Task 1 : 

def summary(state : BatsmanState):
    summary =  f"""
            Strike Rate - {state['sr']}
            Balls per boundary - {state['bpb']}
            Boundary Percentage - {state['boundary_percent']}
   """
    state['summary'] = summary

    return {'summary' : summary}

In [42]:
# Define / Make a graph : 
graph = StateGraph(BatsmanState)

# Add nodes : 
graph.add_node("calculate_sr", calculate_sr)
graph.add_node("calculate_bpb", calculate_bpb)
graph.add_node("calculate_boundary_percent", calculate_boundary_percent)
graph.add_node("summary", summary)


# Add edges :  Parallel Edges 

graph.add_edge(START , 'calculate_sr')
graph.add_edge(START , 'calculate_bpb')
graph.add_edge(START , 'calculate_boundary_percent')

graph.add_edge('calculate_sr' , 'summary')
graph.add_edge('calculate_bpb' , 'summary')
graph.add_edge('calculate_boundary_percent' , 'summary')

graph.add_edge('summary' , END)

# Compile : 

workflow =  graph.compile()


In [43]:
# Execute :

initial_state = {'runs' : 100 , 'balls' : 30 , 'fours' : 7 , 'sixes' : 7} 

workflow.invoke(initial_state)

{'runs': 100,
 'balls': 30,
 'fours': 7,
 'sixes': 7,
 'sr': 333.33333333333337,
 'bpb': 0.02142857142857143,
 'boundary_percent': 70.0,
 'summary': '\n            Strike Rate - 333.33333333333337\n            Balls per boundary - 0.02142857142857143\n            Boundary Percentage - 70.0\n   '}